# 3 — The transformer, component by component

Every tensor shape in this notebook is printed, because shapes are where
seq2seq implementations go wrong.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from nmt.utils.io import project_root, read_json
from nmt.viz.style import use_style

use_style()
print("project root:", ROOT)

In [ ]:
import torch
from nmt.config import ModelConfig
from nmt.model.transformer import TranslationTransformer

torch.manual_seed(0)
config = ModelConfig(vocab_size=16000)
model = TranslationTransformer(config).eval()

breakdown = model.parameter_breakdown()
for key, value in breakdown.items():
    print(f"{key:14s} {value:>12,}")

## Masks

Two logically different things, and conflating them is a classic silent bug.
Throughout this project **`True` means "attend here"**.

In [ ]:
from nmt.model.masking import causal_mask, decoder_mask, padding_mask
from nmt.constants import PAD_ID

tokens = torch.tensor([[5, 6, 7, 8, PAD_ID, PAD_ID]])

print("padding mask (batch, 1, 1, len):", tuple(padding_mask(tokens).shape))
print(padding_mask(tokens)[0, 0, 0].int().tolist())

print("\ncausal mask 5x5:")
print(causal_mask(5).squeeze().int())

print("\ncombined decoder mask, row 3 (may see keys 0..3, but 4-5 are padding):")
print(decoder_mask(tokens)[0, 0, 3].int().tolist())

## Scaled dot-product attention

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Read it as a differentiable dictionary lookup: queries are compared against
keys by dot product, softmax turns the scores into weights, and the output is a
weighted blend of the values.

In [ ]:
from nmt.model.attention import ScaledDotProductAttention

attention = ScaledDotProductAttention()
q = torch.randn(1, 1, 4, 64)
k = torch.randn(1, 1, 6, 64)
v = torch.randn(1, 1, 6, 64)

context, weights = attention(q, k, v)
print("query   ", tuple(q.shape))
print("key     ", tuple(k.shape))
print("context ", tuple(context.shape))
print("weights ", tuple(weights.shape), "  rows sum to", weights.sum(-1).flatten()[:4].tolist())

### Why divide by $\sqrt{d_k}$

The dot product of two $d_k$-dimensional unit-variance vectors has variance
$d_k$, so raw scores grow like $\sqrt{d_k}$. Softmax over widely spread values
saturates, and its gradient $p(1-p)$ collapses to zero — the layer stops
learning.

In [ ]:
import math

d_k = 64
q = torch.randn(1, 1, 1, d_k)
k = torch.randn(1, 1, 500, d_k)

raw = (q @ k.transpose(-2, -1))
scaled = raw / math.sqrt(d_k)

print(f"raw scores    : std {raw.std():.2f}, range [{raw.min():.1f}, {raw.max():.1f}]")
print(f"scaled scores : std {scaled.std():.2f}, range [{scaled.min():.1f}, {scaled.max():.1f}]")

for name, scores in (("raw", raw), ("scaled", scaled)):
    p = torch.softmax(scores, -1)
    entropy = -(p * (p + 1e-12).log()).sum(-1).item()
    print(f"{name:7s}: max weight {p.max():.3f}, entropy {entropy:.2f} nats "
          f"(uniform would be {math.log(500):.2f})")

## Multi-head attention

One softmax attends to essentially one place. Translating "the red house" as
"la casa roja" needs the noun tracked for gender agreement *and* the adjective
tracked for reordering, at the same time.

In [ ]:
from nmt.model.attention import MultiHeadAttention

mha = MultiHeadAttention(512, 8)
x = torch.randn(2, 10, 512)

split = mha._split_heads(x)
print("input          ", tuple(x.shape))
print("after split    ", tuple(split.shape), " <- (batch, heads, len, d_k)")
print("after merge    ", tuple(mha._merge_heads(split).shape))
print("output         ", tuple(mha(x, x, x).shape))
print("\nThe split is a reshape, so h heads cost the same as one.")

## Positional encoding

In [ ]:
from nmt.model.positional import SinusoidalPositionalEncoding

pe = SinusoidalPositionalEncoding(128, max_length=100, dropout=0.0)
table = pe.encoding[0]

print("table shape:", tuple(table.shape))

# Relative position is a linear function of absolute position: the dot product
# between two encodings depends only on their distance apart.
for offset in (1, 2, 5, 10, 20):
    sims = [torch.dot(table[p], table[p + offset]).item() for p in range(10, 40)]
    print(f"offset {offset:2d}: dot product {sum(sims)/len(sims):8.2f} "
          f"(std {torch.tensor(sims).std():.3f})")

The dot product depends almost entirely on the *distance* and barely on the
absolute position — which is what lets attention learn "look three tokens back"
as a single linear map that works everywhere.

## The whole model, and the two invariants that must hold

In [ ]:
source = torch.randint(4, 16000, (2, 9))
target = torch.randint(4, 16000, (2, 7))

logits = model(source, target)
print("source", tuple(source.shape), "target", tuple(target.shape), "-> logits", tuple(logits.shape))

In [ ]:
# INVARIANT 1: the decoder must not see the future.
with torch.no_grad():
    before = model(source, target)
    perturbed = target.clone()
    perturbed[:, 4] = 12345
    after = model(source, perturbed)

print("max change at positions BEFORE the edit:", (before[:, :4] - after[:, :4]).abs().max().item())
print("max change at positions AFTER  the edit:", (before[:, 4:] - after[:, 4:]).abs().max().item())
print("\nThe first must be exactly 0. If it is not, the model is being handed the answer.")

In [ ]:
# INVARIANT 2: a sentence's translation must not depend on its batch-mates.
with torch.no_grad():
    base = model(source, target)
    padded = torch.cat([source, torch.zeros(2, 6, dtype=torch.long)], dim=1)
    extended = model(padded, target)

print("max change from six extra padding columns:", (base - extended).abs().max().item())
print("(float32 round-off only — around 1e-6.)")

## Where the parameters live

In [ ]:
import pandas as pd

rows = []
for name, count in model.parameter_breakdown().items():
    if name in ("total", "trainable"):
        continue
    rows.append({"component": name, "parameters": count,
                 "share": count / model.parameter_breakdown()["total"]})
df = pd.DataFrame(rows).sort_values("parameters", ascending=False)
df["share"] = (100 * df["share"]).round(1).astype(str) + "%"
display(df)

print("\nThe embedding table is shared three ways (encoder input, decoder input,")
print("output projection), so it is counted once. Untying it would add ~17M parameters.")